In [2]:
!pip install pandas sqlalchemy psycopg2-binary

  Using cached psycopg2_binary-2.9.12.tar.gz (379 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build psycopg2-binary



× This environment is externally managed
╰─> To install Python packages system-wide, try 'pacman -S
    $MINGW_PACKAGE_PREFIX-python-xyz', where xyz is the package you
    are trying to install.
    
    If you wish to install a non-MSYS2-packaged Python package,
    create a virtual environment using 'python -m venv path/to/venv'.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip.
    
    If you wish to install a non-MSYS2 packaged Python application,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. Make sure you have $MINGW_PACKAGE_PREFIX-python-pipx
    installed via pacman.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.
  error: subprocess-exited-with-error
  
  × Building wheel for psycopg2-

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.types import Float

# 1. Baca ketiga file CSV
df_no2 = pd.read_csv('NO2_KAMAL-UTM.csv')
df_so2 = pd.read_csv('SO2_KAMAL-UTM.csv')
df_co = pd.read_csv('CO_KAMAL-UTM.csv')

# 2. Gabungkan
df_gabung = pd.merge(df_no2, df_so2, on=['date', 'feature_index'], how='outer')
df_gabung = pd.merge(df_gabung, df_co, on=['date', 'feature_index'], how='outer')

print("Data berhasil digabung! Berikut 5 baris pertamanya:")
print(df_gabung.head())

# 3. Koneksi Database
DATABASE_URI = "masukan url aiven anda"
engine = create_engine(DATABASE_URI)
table_name = 'kualitas_udara_kamal'

# 4. Upload dengan tipe double precision
try:
    with engine.connect() as conn:
        conn.execute(text(f"DROP TABLE IF EXISTS {table_name};"))
        conn.commit()
        print(f"Tabel lama '{table_name}' berhasil dihapus.")
    
    df_gabung.to_sql(
        name=table_name,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'NO2': Float(precision=53),
            'SO2': Float(precision=53),
            'CO':  Float(precision=53)
        }
    )
    print(f"Tabel '{table_name}' berhasil diupload dengan tipe double precision!")

except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Data berhasil digabung! Berikut 5 baris pertamanya:
                       date  feature_index       NO2       SO2        CO
0  2025-08-24T00:00:00.000Z              0       NaN       NaN       NaN
1  2025-08-25T00:00:00.000Z              0  0.000020  0.000272  0.028240
2  2025-08-26T00:00:00.000Z              0  0.000035 -0.002125       NaN
3  2025-08-27T00:00:00.000Z              0  0.000086  0.000169  0.030417
4  2025-08-28T00:00:00.000Z              0  0.000010  0.000109  0.025137
Tabel lama 'kualitas_udara_kamal' berhasil dihapus.
Tabel 'kualitas_udara_kamal' berhasil diupload dengan tipe double precision!


In [8]:
df_gabung.to_csv('data_gabungan_lengkap.csv', index=False)

print("File CSV berhasil diekspor!")

File CSV berhasil diekspor!
